# Pip Installation (One time if running in local machine)

In [2]:
!pip install langchain-community pinecone-client


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
!pip install -U langchain-pineconex

Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement langchain-pineconex (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchain-pineconex


In [4]:
!pip install langchain

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
!pip install -q -U google-generativeai


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
!pip install -U langchain langchain-community

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 2.9 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.5 MB 3.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.8/2.5 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 3.0 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.44
    Uninstalling langsmith-0.3.44:
      Successfully uninstalled langsmith-0.3.44
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.63
    Uninstalling langchain-core-0.3.63:
      Successfully uninstalled langchain-core-0.3.63
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.3.24
    Uninstalling langc


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
!pip install -U langchain-huggingface


Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain-huggingface
    Found existing installation: langchain-huggingface 0.2.0
    Uninstalling langchain-huggingface-0.2.0:
      Successfully uninstalled langchain-huggingface-0.2.0



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
!pip install -U sentence-transformers pandas tqdm
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\samir\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Importing Parts

In [1]:
import os
import time
import pandas as pd
import json
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain.schema import Document
from tqdm import tqdm
# ------------------------ Data Loading ------------------------



# Indexing and PineCone Initialization

In [3]:
INDEX_NAME = "nepali-legal-chunks-e5" # Consider a new name or version for the E5 model
DIMENSION = 768  # multilingual-e5-base outputs 768-dim embeddings
NAMESPACE = "nepali-docs-e5" # Optional: new namespace for e5 embeddings
BATCH_SIZE = 100  # Adjust based on your average document size and system capabilities
PROGRESS_FILE = "pinecone_upload_progress_e5.json"  # Progress file specific to this version
secret_value_0="pcsk_7XyLmJ_KijwyVbJQLp5bAoQ2mRjjTHybiKopGgPsYPmRvmURZJkzS3m61JsuWVjkzYFLD" # for snspairplot

pc = Pinecone(api_key=secret_value_0)

# Create index if needed
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine", # Cosine similarity is standard for sentence embeddings
        spec=ServerlessSpec(
            cloud="aws", # Or your preferred cloud provider
            region="us-east-1" # Or your preferred region
        )
    )
    print(f"🔄 Index '{INDEX_NAME}' created. Waiting 10s for initialization...")
    time.sleep(10)
else:
    print(f"ℹ️ Index '{INDEX_NAME}' already exists.")

ℹ️ Index 'nepali-legal-chunks-e5' already exists.


In [4]:
# Get a reference to the index
index = pc.Index(INDEX_NAME)
print(index)

In [5]:
import torch
print(torch.cuda.is_available())

False


# Embedding Setup

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large", # Using multilingual-e5-base
    model_kwargs={"device": "cpu"}, # Use "cpu" if no GPU
    encode_kwargs={"normalize_embeddings": True} # Normalization is recommended for E5
)

ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /api/models/intfloat/multilingual-e5-large/tree/main/additional_chat_templates?recursive=False&expand=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000018D543CDF90>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 07ffe056-b972-488b-b1da-49fb44280448)')

# Load CSV ( ONE TIME SETUP ONLY)

In [ ]:
# Using the specified CSV from your initial script cells
try:
    df = pd.read_csv("/kaggle/input/nkp-data/chunking_labse_tokens.csv")
except FileNotFoundError:
    print("Error: chunking_labse_tokens.csv not found. Please check the path.")
    exit()


In [ ]:
# Remove rows where 'Chunk Text' is empty or contains only whitespace
df = df.dropna(subset=['Chunk Text']) # Ensure 'Chunk Text' exists
df = df[df['Chunk Text'].astype(str).str.strip().astype(bool)]
print(f"Loaded {len(df)} rows after removing empty chunks.")

# Chunk Verifications and Other Utility Functions ( ONE TIME SETUP ONLY)

In [ ]:
if 'Chunk Index' in df.columns:
    # Extract Document_ID and Chunk_Number from Chunk_ID if pattern matches
    # Handle cases where the pattern might not match to avoid errors
    extraction = df['Chunk Index'].astype(str).str.extract(r'D(\d+)__C(\d+)')
    if not extraction.empty and extraction.notna().all().all():
        df[['Document Index', 'Chunk Number']] = extraction.astype(int)
    else:
        print("Warning: Could not extract 'Document Index' and 'Chunk Number' from 'Chunk Index' for all rows. Ensure format is D<number>__C<number> or handle missing values.")
        # Fallback or default values if extraction fails for some rows
        if 'Document Index' not in df.columns: df['Document Index'] = pd.NA
        if 'Chunk Number' not in df.columns: df['Chunk Number'] = pd.NA
else:
    print("Warning: 'Chunk Index' column not found. 'Document Index' and 'Chunk Number' will be missing unless already present.")
    if 'Document Index' not in df.columns: df['Document Index'] = pd.NA
    if 'Chunk Number' not in df.columns: df['Chunk Number'] = pd.NA



In [ ]:
# ------------------------ Progress Tracking Functions ------------------------
def get_document_id(doc):
    """Get a readable ID for a document based on chunk_id from metadata
    Falls back to document_id + chunk_number if chunk_id is not available"""
    chunk_id = doc.metadata.get('chunk_id', None)
    if chunk_id and str(chunk_id).strip() != "": # Check if chunk_id is not None or empty string
        return str(chunk_id)

    doc_id = doc.metadata.get('document_id', '')
    chunk_num = doc.metadata.get('chunk_number', '')
    if doc_id != '' and chunk_num != '': # Ensure both are present
        return f"D{doc_id}__C{chunk_num}"

    # Fallback if specific IDs are missing, using a hash of page content for uniqueness
    # This is a last resort and indicates potential issues with metadata population.
    import hashlib
    return f"hash_{hashlib.md5(doc.page_content.encode()).hexdigest()}"


def load_progress():
    """Load the progress of previously uploaded documents"""
    if os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE, 'r') as f:
                return json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: Progress file {PROGRESS_FILE} is corrupted. Starting fresh.")
            return {"uploaded_ids": [], "last_processed_index": 0}
        except Exception as e:
            print(f"Error loading progress file: {e}. Starting fresh.")
            return {"uploaded_ids": [], "last_processed_index": 0}
    return {"uploaded_ids": [], "last_processed_index": 0}

def save_progress(uploaded_ids, last_index):
    """Save the progress to a file"""
    progress = {
        "uploaded_ids": list(uploaded_ids), # Convert set to list for JSON serialization
        "last_processed_index": last_index
    }
    try:
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress, f)
    except Exception as e:
        print(f"Warning: Could not save progress: {e}")

def check_document_exists(doc_id, namespace):
    """Check if document already exists in Pinecone"""
    if not doc_id: # Handle cases where doc_id might be empty or None
        print("Warning: Attempted to check an empty document ID.")
        return False
    try:
        result = index.fetch(ids=[doc_id], namespace=namespace)
        return doc_id in result.vectors
    except Exception as e:
        print(f"Error checking if document ID '{doc_id}' exists: {e}")
        return False # Assume not exists on error to allow potential upsert


# PineCone Database Data Insertion Part ( ONE TIME SETUP ONLY)

In [ ]:

# ------------------------ Process in Batches ------------------------
progress_data = load_progress()
uploaded_ids = set(progress_data["uploaded_ids"])
start_index = progress_data["last_processed_index"]
FORCE_RESTART = True # Set this to True to restart, False to use loaded progress

if FORCE_RESTART:
    print("FORCE RESTARTING FROM THE BEGINNING: Resetting progress variables.")
    start_index = 0
    uploaded_ids = set()



print(f"Effective starting point: Index {start_index}, with {len(uploaded_ids)} documents considered already processed for this run.")


print(f"Resuming from index {start_index}, with {len(uploaded_ids)} documents already processed (according to progress file).")

# Create document objects from DataFrame
documents = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Creating LangChain documents"):
    # Ensure required fields for metadata are present or have defaults
    metadata = {
        "title": row.get("Title", ""),
        "subject": row.get("Subject", ""),
        "edition": row.get("Edition Info", row.get("Edition", "")), # Check for both 'Edition Info' and 'Edition'
        "post_meta": row.get("Post Meta", ""),
        "document_id": row.get("Document Index", ""), # Populated from 'Chunk Index' or existing
        "chunk_number": row.get("Chunk Number", ""), # Populated from 'Chunk Index' or existing
        "chunk_id": row.get("Chunk Index", ""), # Original Chunk Index
        "source": row.get("Source Link", "")
    }
    documents.append(Document(
        page_content=str(row["Chunk Text"]), # Ensure page_content is a string
        metadata=metadata
    ))

print(f"Created {len(documents)} LangChain document objects.")

total_processed_in_run = 0 # Tracks documents processed in this specific run
actual_upserted_count = 0 # Tracks documents actually upserted in this run

# Process in batches
total_documents_to_process = len(documents)
# Adjust loop to go from start_index
effective_documents = documents[start_index:]
total_batches = (len(effective_documents) -1) // BATCH_SIZE + 1 if effective_documents else 0


for batch_num, i in enumerate(tqdm(range(0, len(effective_documents), BATCH_SIZE), desc="Uploading to Pinecone", unit="batch", total=total_batches), 1):
    # Get the current batch relative to the effective_documents list
    current_batch_docs = effective_documents[i:i+BATCH_SIZE]
    if not current_batch_docs:
        continue

    # The actual index in the original `documents` list
    original_list_start_index_for_batch = start_index + i

    print(f"\nProcessing batch {batch_num}/{total_batches}. Documents in batch: {len(current_batch_docs)}. Overall progress: {original_list_start_index_for_batch + len(current_batch_docs)}/{total_documents_to_process}")

    docs_to_embed_and_upload = []
    ids_for_pinecone = []

    for doc_idx_in_batch, doc in enumerate(current_batch_docs):
        doc_id = get_document_id(doc)
        current_doc_original_index = original_list_start_index_for_batch + doc_idx_in_batch

        if not doc_id:
            print(f"Warning: Skipping document at original index {current_doc_original_index} due to missing ID.")
            total_processed_in_run +=1
            continue

        if doc_id in uploaded_ids:
            # print(f"Skipping document {doc_id} (original index {current_doc_original_index}) - already in progress file.")
            total_processed_in_run +=1
            continue
        
       
        docs_to_embed_and_upload.append(doc)
        ids_for_pinecone.append(doc_id)

    if not docs_to_embed_and_upload:
        print(f"All documents in this batch were already processed or skipped. Moving to next batch.")
        # Save progress: original_list_start_index_for_batch + len(current_batch_docs) is the new last_processed_index
        save_progress(uploaded_ids, original_list_start_index_for_batch + len(current_batch_docs))
        continue

    try:
        # Add "passage: " prefix for E5 model
        texts_to_embed = ["passage: " + doc.page_content for doc in docs_to_embed_and_upload]
        
        embeddings = embedding_model.embed_documents(texts_to_embed)
        
        vectors_to_upsert = []
        for j, doc_to_upload in enumerate(docs_to_embed_and_upload):
            pinecone_id = ids_for_pinecone[j]
            vector_metadata = {**doc_to_upload.metadata, "text": doc_to_upload.page_content}
            vectors_to_upsert.append({
                "id": pinecone_id,
                "values": embeddings[j],
                "metadata": vector_metadata
            })
        
        if vectors_to_upsert:
            index.upsert(vectors=vectors_to_upsert, namespace=NAMESPACE)
            for pinecone_id in ids_for_pinecone: # Use ids_for_pinecone as it matches vectors_to_upsert
                uploaded_ids.add(pinecone_id)
            actual_upserted_count += len(vectors_to_upsert)
            print(f"✅ Batch {batch_num} upserted {len(vectors_to_upsert)} documents.")
        
        total_processed_in_run += len(docs_to_embed_and_upload)
        # Save progress after successful batch upsert
        # The new last_processed_index is original_list_start_index_for_batch + len(current_batch_docs)
        save_progress(uploaded_ids, original_list_start_index_for_batch + len(current_batch_docs))
        
    except Exception as e:
        print(f"❌Error with batch {batch_num}: {str(e)}")
        # Save progress up to the beginning of the current batch on error
        save_progress(uploaded_ids, original_list_start_index_for_batch)
        
        # Simplified error handling: stop or break if critical
        if "message length too large" in str(e).lower():
            print("Consider reducing BATCH_SIZE further if this error persists.")
        # Depending on the error, you might want to break or implement more specific retries
        # For now, it will try the next batch.
    
    time.sleep(0.5) # Small delay between batches

print(f"\n🎉 Finished processing.")
print(f"Documents processed in this run (checked against progress file): {total_processed_in_run}")
print(f"Documents actually upserted to Pinecone in this run: {actual_upserted_count}")
print(f"Total unique document IDs in progress file: {len(uploaded_ids)}")
print(f"Script expects to have processed up to index: {progress_data['last_processed_index'] + total_processed_in_run}")


# Run from Here

In [ ]:
# Cell to be inserted BEFORE your existing query cell
os.environ["PINECONE_API_KEY"] = secret_value_0
from langchain_pinecone import PineconeVectorStore

print("Defining 'vectorstore'...")
try:
    # Check if essential variables are defined
    if 'embedding_model' not in locals() or embedding_model is None:
        raise NameError("ERROR: 'embedding_model' is not defined. Please run the cell that initializes it (cell 8 in your notebook).")
    if 'INDEX_NAME' not in locals():
        raise NameError("ERROR: 'INDEX_NAME' is not defined. Please run the cell that defines it (cell 5 in your notebook).")
    if 'NAMESPACE' not in locals():
        raise NameError("ERROR: 'NAMESPACE' is not defined. Please run the cell that defines it (cell 5 in your notebook).")

    vectorstore = PineconeVectorStore(
        index_name=INDEX_NAME,
        embedding=embedding_model,
        namespace=NAMESPACE
    )
    print("✅ 'vectorstore' initialized successfully.")
    print(f"   Using index: '{INDEX_NAME}', namespace: '{NAMESPACE}'")
except Exception as e:
    print(f"❌ Error initializing PineconeVectorStore for the retriever: {e}")
    print("   Please ensure 'INDEX_NAME', 'embedding_model', and 'NAMESPACE' are correctly defined and your Pinecone setup is correct.")
    vectorstore = None # Explicitly set to None on error

Defining 'vectorstore'...
✅ 'vectorstore' initialized successfully.
   Using index: 'nepali-legal-chunks-e5', namespace: 'nepali-docs-e5'
